In [1]:
import pandas as pd
import numpy as np

In [2]:
df_train_raw = pd.read_csv('../tiktok/train.csv')
df_test_raw = pd.read_csv('../tiktok/test.csv')

display(df_train_raw.head(3))

,log_event_id,vid_key,batch_hour_seq,sys_traffic_src,usr_key,attr_tier,attr_tenure_m,snd_key,attr_bpm,cat_code,pub_hour_utc,vid_len_sec,val_watch_sec,val_shares_cnt,val_swipe_pct,post_payout_usd,target_fyp_viral
0,EVT_101935,VID_00194,6,ORGANIC,USR_0141,Top_Partner,51,SND_0041,102,Storytime,11,15,5.3,20,0.635,0.0,0
1,EVT_106494,VID_00650,5,ORGANIC,USR_0216,Cuenta_Nueva,46,SND_0100,137,Storytime,5,15,-999.0,21,0.463,0.0,0
2,EVT_100360,VID_00037,1,ORGANIC,USR_0058,Cuenta_Nueva,28,SND_0127,154,Gaming,23,45,23.5,21,0.450,0.0,0


In [13]:
df_dim_creadores = df_train_raw[['usr_key', 'attr_tier', 'attr_tenure_m']].drop_duplicates().reset_index(drop=True)
df_dim_audios = df_train_raw[['snd_key', 'attr_bpm']].drop_duplicates().reset_index(drop=True)

df_fact_telemetria = df_train_raw.drop(columns=['attr_tier', 'attr_tenure_m', 'attr_bpm'])

display(df_dim_creadores.head(3))
display(df_dim_audios.head(3))

,usr_key,attr_tier,attr_tenure_m
0,USR_0141,Top_Partner,51
1,USR_0216,Cuenta_Nueva,46
2,USR_0058,Cuenta_Nueva,28


,snd_key,attr_bpm
0,SND_0041,102
1,SND_0100,137
2,SND_0127,154


In [15]:
df_organico = df_fact_telemetria[df_fact_telemetria["sys_traffic_src"] == "ORGANIC"].replace(-999, np.nan)
df_organico['val_watch_sec'] = df_organico['val_watch_sec'].replace(-999, np.nan)
df_organico.loc[df_organico['val_watch_sec'] > df_organico['vid_len_sec'],'val_watch_sec'] = np.nan

df_cubo_olap = df_organico.groupby("vid_key").agg({
    "usr_key": "first",
    "snd_key": "first",
    "cat_code": "first",
    "pub_hour_utc": "first",
    "vid_len_sec": "first",
    "post_payout_usd": "first",
    "target_fyp_viral": "first",
    "val_shares_cnt": "sum", # se suma el conteo de horas
    "val_watch_sec": "mean", # se promedia el tiempo de vistas
    "val_swipe_pct": "mean" # se promedia los swipes
}).reset_index()

display(df_cubo_olap.head(3))


,vid_key,usr_key,snd_key,cat_code,pub_hour_utc,vid_len_sec,post_payout_usd,target_fyp_viral,val_shares_cnt,val_watch_sec,val_swipe_pct
0,VID_00001,USR_0032,SND_0028,Humor_Meme,20,30,127.12,1,199,19.942,0.3281
1,VID_00003,USR_0109,SND_0071,Humor_Meme,20,15,0.00,0,122,4.089,0.7393
2,VID_00004,USR_0238,SND_0063,Storytime,1,15,0.00,0,82,7.948,0.4906


In [16]:
df_test = df_test_raw.copy()

df_test['val_watch_sec'] = df_test['val_watch_sec'].replace(-999, np.nan)
df_test.loc[df_test['val_watch_sec'] > df_test['vid_len_sec'], 'val_watch_sec'] = np.nan

df_cubo_test = df_test.groupby('vid_key').agg({
    'usr_key': 'first',
    'snd_key': 'first',
    'cat_code': 'first',
    'pub_hour_utc': 'first',
    'vid_len_sec': 'first',
    'val_shares_cnt': 'sum',
    'val_watch_sec': 'mean',
    'val_swipe_pct': 'mean'
}).reset_index()

In [8]:
class EvaluadorKaggleFYP:
    @staticmethod
    def auditar_estrella(df_fact, df_dim_creadores, df_dim_audios):
        print("🔍 Evaluando Checkpoint 1: Esquema Estrella...")
        if not df_dim_creadores['usr_key'].is_unique: return "❌ ERROR: 'usr_key' tiene duplicados."
        if not df_dim_audios['snd_key'].is_unique: return "❌ ERROR: 'snd_key' tiene duplicados."
        
        columnas_prohibidas = ['attr_tier', 'attr_tenure_m', 'attr_bpm']
        for col in columnas_prohibidas:
            if col in df_fact.columns: return f"❌ ERROR: La columna descriptiva '{col}' sigue en fact_telemetria."
                
        print("✅ CHECKPOINT 1 SUPERADO: Normalización exitosa.\n")
        return True

    @staticmethod
    def auditar_cubo_olap(df_cubo):
        print("🔍 Evaluando Checkpoint 2: Cubo OLAP...")
        # En el set de entrenamiento hay ~700 videos orgánicos, en el de test hay ~300. 
        # Si la llave es única, el roll-up se hizo bien.
        if 'vid_key' not in df_cubo.columns or not df_cubo['vid_key'].is_unique:
            return "❌ ERROR: 'vid_key' no es único. No agrupaste a nivel de video."
            
        print("✅ CHECKPOINT 2 SUPERADO: Granularidad OLAP correcta.\n")
        return True

evaluador = EvaluadorKaggleFYP()

In [9]:
print(evaluador.auditar_estrella(df_fact_telemetria, df_dim_creadores, df_dim_audios))
print(evaluador.auditar_cubo_olap(df_cubo_olap))

🔍 Evaluando Checkpoint 1: Esquema Estrella...
✅ CHECKPOINT 1 SUPERADO: Normalización exitosa.

True
🔍 Evaluando Checkpoint 2: Cubo OLAP...
✅ CHECKPOINT 2 SUPERADO: Granularidad OLAP correcta.

True


In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# --- TU CÓDIGO AQUÍ: Feature Engineering  ---
tier = {
    'Cuenta_Nueva': 1,
    'Afiliado' : 2,
    'Socio_Verificado': 3,
    'Top_Partner': 4
}

columasBasura = ['vid_key', 'usr_key', 'snd_key', 'attr_tier', 'val_watch_sec', 'vid_len_sec', 'post_payout_usd', 'target_fyp_viral']
df_ml_train = df_cubo_olap.merge(df_dim_creadores, on='usr_key', how='left')
df_ml_train = df_ml_train.merge(df_dim_audios, on='snd_key', how='left')
df_ml_train['tier_encoded'] = df_ml_train['attr_tier'].map(tier)
df_ml_train = pd.get_dummies(df_ml_train, columns=['cat_code'], dtype=int)
df_ml_train['hook_rate'] = df_ml_train['val_watch_sec'] / df_ml_train['vid_len_sec']


dim_creadores_test = df_test_raw[['usr_key', 'attr_tier', 'attr_tenure_m']].drop_duplicates()
dim_audios_test = df_test_raw[['snd_key', 'attr_bpm']].drop_duplicates()

df_ml_test = df_cubo_test.merge(dim_creadores_test, on='usr_key', how='left')
df_ml_test = df_ml_test.merge(dim_audios_test, on='snd_key', how='left')
df_ml_test['tier_encoded'] = df_ml_test['attr_tier'].map(tier)
df_ml_test = pd.get_dummies(df_ml_test, columns=['cat_code'], dtype=int)
df_ml_test['hook_rate'] = df_ml_test['val_watch_sec'] / df_ml_test['vid_len_sec']


X_train = df_ml_train.drop(columns=columasBasura)
y_train = df_ml_train['target_fyp_viral']
X_test_limpio = df_ml_test.drop(columns=columasBasura, errors='ignore')

print(X_train.shape)


# 1. Entrenar el Modelo
modelo = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
modelo.fit(X_train, y_train)

# 2. Predecir sobre el set de prueba ciego
predicciones = modelo.predict(X_test_limpio)

# 3. Exportar a Kaggle
submission = pd.DataFrame({
    'vid_key': df_ml_test['vid_key'], # Asegúrate de que tu matriz final aún tenga el ID o guárdalo antes
    'target_fyp_viral': predicciones
})
submission.to_csv('submission.csv', index=False)
print("¡Archivo listo! Súbelo al Leaderboard.")

df_answer = pd.read_csv("submission.csv")

df_answer.describe()

(696, 12)
¡Archivo listo! Súbelo al Leaderboard.


,target_fyp_viral
count,360.000000
mean,0.152778
std,0.360274
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,1.000000
